In [ ]:
import sys
sys.path.append('../')
sys.path.append("../../")
import numpy as np
import matplotlib.pyplot as plt
import os
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import torch
import torch.nn.functional as F
from pydose_rt import ModelConfig
from pydose_rt import DoseEngine
from pydose_rt.engine.utils.plotting import *
from pydose_rt.utils.kernel import *
from pydose_rt.engine.utils.grad_monitor import GradMonitor
from torch.utils.data import DataLoader  # PyTorch DataLoader
from pydose_rt.engine.data_augment import DataGenerator
from pydose_rt.engine.config import config as PARAMS
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from IPython.display import clear_output
import torch.nn.functional as F
import torch
from skimage import measure
from pydose_rt.engine.loss import (
    dose_loss,
    leafs_loss,
    mus_loss,
    jaws_loss,
)

In [ ]:
def overlay_mask_outline(mask_slice, color="red", linewidth=2):
    for contour in measure.find_contours(mask_slice, 0.5):
        plt.plot(contour[:, 1], contour[:, 0], color=color, linewidth=linewidth)

def dilate_3d(mask, iterations, kernel_size=3):
    """
    Perform binary dilation in 3D using 3D convolution.
    mask: (B, D, H, W) or (1, D, H, W)
    """
    B, D, H, W = mask.shape
    # Add channel dimension for convolution
    mask = mask.unsqueeze(1).float()  # shape: (B, 1, D, H, W)

    kernel = torch.ones((1, 1, kernel_size, kernel_size, kernel_size), device=mask.device)

    for _ in range(iterations):
        mask = F.conv3d(mask, kernel, padding=kernel_size // 2)
        mask = (mask > 0).float()  # keep it binary

    return mask.squeeze(1)  # back to shape: (B, D, H, W)

def scale_loss(loss, weight):
    return loss * weight

def create_ring_mask_torch(ptv_mask, inner_iter=0, outer_iter=4):
    """
    Create a ring mask by subtracting two dilations of the PTV mask.
    
    ptv_mask: torch.Tensor of shape (1, D, H, W), binary mask
    inner_iter: inner dilation iterations (~1cm)
    outer_iter: outer dilation iterations (~2cm)
    """
    assert ptv_mask.dim() == 4, "Expected shape (1, D, H, W)"
    
    dilated_inner = dilate_3d(ptv_mask, inner_iter)
    dilated_outer = dilate_3d(ptv_mask, outer_iter)
    
    ring_mask = (dilated_outer > 0) & (dilated_inner == 0)
    return ring_mask  # shape: (1, D, H, W)

In [ ]:
config = ModelConfig(preset="lund-probe", number_of_cps=240, downsampling_factor=(2,2,2))

In [ ]:
if (os.path.exists('/mnt/')):
  data_path = "/media/bolo/f4616a95-e470-4c0f-a21e-a75a8d283b9e/RAW/lund-probe-processed/" # '/media/bolo/f4616a95-e470-4c0f-a21e-a75a8d283b9e/DATASETS/ARTP/'

gen = DataGenerator(data_path, "training", None, None, shuffle=False, batch_size=1, downsampling_factor=(1,1,1), constraints=PARAMS.constraints_lund_probe)
val_loader = DataLoader(
    gen,
    batch_size=gen.batch_size,  # Use same batch size or 1 for validation
    shuffle=gen.shuffle,
    num_workers=0,
    pin_memory=True,
)
for i, batch_data in enumerate(val_loader):
  x, y_dose, masks, region_weights, constraints_batch = batch_data
  break

In [ ]:
mask_target = torch.tensor(masks[0, 0, ...].expand(1, -1, -1, -1), dtype=torch.float32, device=device) > 0
mask_external = torch.tensor(masks[:, -1] > 0, dtype=torch.bool, device=device)
mask_oar = torch.tensor(torch.sum(masks[0, 1:-1, ...], 0).expand(1, -1, -1, -1), dtype=torch.float32, device=device) > 0
dose_target = torch.tensor(y_dose.expand(1, -1, -1, -1), device=device)
masks_torch = []
for i in range(masks.shape[1]):
    masks_torch.append(masks[0, i, ...].expand(1, -1, -1, -1))
dose_layer = DoseEngine(config, 5)
dose_layer.train()
ring_mask = create_ring_mask_torch(mask_target).clone().detach().to(device=mask_target.device)

In [ ]:
x = x.to(config.device)
y_dose = y_dose.to(config.device)
masks = masks.to(config.device)
region_weights = region_weights.to(config.device)

In [ ]:
watch = [
    "beam_wise_conv_layer",
    "fluence_map_layer",
    "fluence_volume_layer",
]

monitor = GradMonitor(modules_to_watch=watch).install(dose_layer)

In [ ]:
def print_results(raw_losses, y_dose, y_mlc, mus, jaws, best_results, dose_pred, masks, mae_loss):
    def _hide_ticks(ax):
        ax.set_xticks([])
        ax.set_yticks([])
        ax.tick_params(bottom=False, left=False)

    scale = np.max(np.abs(y_mlc.grad.cpu().detach().numpy()))
    mus_max = mus.cpu().detach().numpy().max()
    aspect_ratio = 5.0
    dose_max = 50.0
    alpha = 0.1
    
    fig = plt.figure(figsize=(12, 10))
    fig.suptitle(f"MAE - {str(mae_loss)}Gy\nTest #{len(best_results)}: {[str(np.round(v, 4)) for v in raw_losses]}")
    
    # 5–6
    ax = plt.subplot(20,1,1)
    ax.set_title('Jaws (centers)')
    ax.imshow(jaws.cpu().detach().numpy()[0, 0:1, :],
              vmin=0.0, vmax=mus_max, cmap='gray',
              aspect=aspect_ratio, interpolation='none')
    ax.imshow(jaws.grad.cpu().detach().numpy()[0, 0:1, :],
              cmap='coolwarm', vmin=-scale, vmax=scale,
              aspect=aspect_ratio, interpolation='none', alpha=alpha)
    _hide_ticks(ax)

    # 7–8
    ax = plt.subplot(20,1,2)
    ax.set_title('Jaws (widths)')
    ax.imshow(jaws.cpu().detach().numpy()[0, 1:2, :],
              vmin=0.0, vmax=mus_max, cmap='gray',
              aspect=aspect_ratio, interpolation='none')
    ax.imshow(jaws.grad.cpu().detach().numpy()[0, 1:2, :],
              cmap='coolwarm', vmin=-scale, vmax=scale,
              aspect=aspect_ratio, interpolation='none', alpha=alpha)
    _hide_ticks(ax)

    # 1–2
    ax = plt.subplot(20,1,3)
    ax.set_title('MLCs (centers)')
    im = ax.imshow(np.transpose(y_mlc.cpu().detach().numpy()[0, 0, :, :]),
                   cmap='gray', vmin=0, vmax=1,
                   aspect=aspect_ratio / config.number_of_leaf_pairs, interpolation='none')
    im = ax.imshow(np.transpose(y_mlc.grad.cpu().detach().numpy()[0, 0, :, :]),
                   cmap='coolwarm', vmin=-scale, vmax=scale,
                   aspect=aspect_ratio / config.number_of_leaf_pairs, interpolation='none', alpha=alpha)
    _hide_ticks(ax)

    # 3–4
    ax = plt.subplot(20,1,4)
    ax.set_title('MLCs (widths)')
    ax.imshow(np.transpose(y_mlc.cpu().detach().numpy()[0, 1, :, :]),
              cmap='gray', vmin=0, vmax=1,
              aspect=aspect_ratio / config.number_of_leaf_pairs, interpolation='none')
    ax.imshow(np.transpose(y_mlc.grad.cpu().detach().numpy()[0, 1, :, :]),
              cmap='coolwarm', vmin=-scale, vmax=scale,
              aspect=aspect_ratio / config.number_of_leaf_pairs, interpolation='none', alpha=alpha)
    _hide_ticks(ax)

    # 9–10
    ax = plt.subplot(20,1,5)
    ax.set_title('MUs')
    ax.imshow(mus.cpu().detach().numpy(),
              vmin=0.0, vmax=mus_max, cmap='gray',
              aspect=aspect_ratio, interpolation='none')
    ax.imshow(mus.grad.cpu().detach().numpy(),
              cmap='coolwarm', vmin=-scale, vmax=scale,
              aspect=aspect_ratio, interpolation='none', alpha=alpha)
    ax.set_xlabel("Control points")
    _hide_ticks(ax)

    # 11
    ax = plt.subplot(423)
    ax.set_title('Dose distribution (cropped)')
    ax.imshow(dose_pred.cpu().detach().numpy()[0, :, :, 44],
             cmap='jet', vmin=0.0, vmax=dose_max, aspect=0.2)
    _hide_ticks(ax)
    for idx, (key, value) in enumerate(list(PARAMS.structure_names.items())[:-1]):
        roi_name = value
        roi = masks[idx]
        overlay_mask_outline(roi.cpu().numpy()[0, :, :, 44], color=PARAMS.roi_colors[key])

    # 12
    ax = plt.subplot(424)
    ax.set_title('Dose distribution (cropped)')
    ax.imshow(np.transpose(dose_pred.cpu().detach().numpy()[0, 64:198, 128, :]),
             cmap='jet', vmin=0.0, vmax=dose_max, aspect=0.2)
    _hide_ticks(ax)
    for idx, (key, value) in enumerate(list(PARAMS.structure_names.items())[:-1]):
        roi_name = value
        roi = masks[idx]
        overlay_mask_outline(np.transpose(roi.cpu().numpy()[0, 64:198, 128, :]), color=PARAMS.roi_colors[key])

    # 11
    ax = plt.subplot(425)
    ax.set_title('Dose distribution (cropped)')
    ax.imshow(y_dose.cpu().detach().numpy()[0, :, :, 44],
             cmap='jet', vmin=0.0, vmax=dose_max, aspect=0.2)
    _hide_ticks(ax)
    for idx, (key, value) in enumerate(list(PARAMS.structure_names.items())[:-1]):
        roi_name = value
        roi = masks[idx]
        overlay_mask_outline(roi.cpu().numpy()[0, :, :, 44], color=PARAMS.roi_colors[key])

    # 12
    ax = plt.subplot(426)
    ax.set_title('Dose distribution (cropped)')
    ax.imshow(np.transpose(y_dose.cpu().detach().numpy()[0, 64:198, 128, :]),
             cmap='jet', vmin=0.0, vmax=dose_max, aspect=0.2)
    _hide_ticks(ax)
    for idx, (key, value) in enumerate(list(PARAMS.structure_names.items())[:-1]):
        roi_name = value
        roi = masks[idx]
        overlay_mask_outline(np.transpose(roi.cpu().numpy()[0, 64:198, 128, :]), color=PARAMS.roi_colors[key])

    # 13–15 (DVH)
    ax = plt.subplot(414)
    for idx, (key, value) in enumerate(PARAMS.structure_names.items()):
        roi_name = value
        roi = masks[idx]

        dose_values = dose_pred[roi > 0.0].cpu().detach().numpy()
        bins = np.linspace(0, dose_max, 1000)
        hist, bin_edges = np.histogram(dose_values, bins=bins, density=False)
        cumulative_hist = np.cumsum(hist[::-1])[::-1]
        cumulative_hist_normalized = cumulative_hist / cumulative_hist.max()

        ax.plot(bin_edges[:-1], cumulative_hist_normalized,
                label=roi_name, color=PARAMS.roi_colors[key])

    ax.set_xlabel("Dose (Gy)")
    ax.set_ylabel("Volume Fraction")
    ax.set_title("Dose Volume Histogram (DVH)")
    ax.grid(True)
    ax.legend(loc="lower left")

    clear_output(wait=True)
    fig.tight_layout(rect=[0, 0, 1, 0.97])  # keep space for the suptitle
    plt.show()


In [ ]:
print_stuff = 0
loss_plot = 1.0
best_results = []
n_tests = 200
patience_thr = 20

oar_dose = 10.0

def compute_loss(dose_pred, mus, leafs, jaws, weights):
    (
        loss_lower_bound_gy,
        loss_higher_bound_gy,
        loss_lower_bound_target,
        loss_higher_bound_target,
        l2_loss_oars_and_background,
    ) = dose_loss(x, dose_pred, constraints_batch, masks, region_weights, None)
    mu_rate_loss, mu_complexity_loss = mus_loss(mus, config)
    leaf_reg_loss, leaf_complexity_loss = leafs_loss(leafs, config)
    jaw_opening_loss, jaw_complexity_loss = jaws_loss(jaws, config)
    all_losses = [
        scale_loss(loss_lower_bound_gy, weights[0]),
        scale_loss(loss_higher_bound_gy, weights[1]),
        scale_loss(loss_lower_bound_target, weights[2]),
        scale_loss(loss_higher_bound_target, weights[3]),
        scale_loss(l2_loss_oars_and_background, weights[4]),
        scale_loss(mu_rate_loss, weights[5]),
        scale_loss(mu_complexity_loss, weights[6]),
        scale_loss(leaf_reg_loss, weights[7]),
        scale_loss(leaf_complexity_loss, weights[8]),
        scale_loss(jaw_opening_loss, weights[9]),
        scale_loss(jaw_complexity_loss, weights[10]),
    ]
    return all_losses

for test_i in range(n_tests):
    current_res = [np.inf]
    patience = 0
    y_mlc_init = (0.5 * torch.rand((1, 2, config.number_of_cps, config.number_of_leaf_pairs), dtype=torch.float32, device=device))
    y_mlc_init[:, 0, :, :] = 0.5
    y_mlc = y_mlc_init.clone().detach().requires_grad_(True)
    jaws_init = 2.0 * torch.rand((1, 2, config.number_of_cps), dtype=torch.float32, device=device)
    jaws_init[:, 0, :] = 0.5
    jaws_init[:, 1, :] = 1.0
    jaws = jaws_init.clone().detach().requires_grad_(True)
    mus_init = (1000.0 * torch.ones((1, config.number_of_cps), dtype=torch.float32, device=device) + 0.5) / config.number_of_cps
    mus = mus_init.clone().detach().requires_grad_(True)
    weights = 10.0**np.random.randint(-3, 2, size=11) #  [0.01,  1.  ,  0.01, 10.  ,  0.01,  0.01,  0.01, 10.  , 10.  , 10.  ,  1.] Something buggy here
    weights[0] = 1.0
    weights[1] = 1.0
    latest = {"raw_losses": None, "loss_val": None, "dose_pred": None}

    for take_every in [1]:
        optimizer = torch.optim.Adam([y_mlc, mus, jaws], lr=1e-1, weight_decay=1e-4)
        # scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.95, patience=20)

        while patience < patience_thr:
            dose_layer.train()
            
            def closure():
                optimizer.zero_grad(set_to_none=True)

                # Forward
                dose_pred = dose_layer(y_mlc, mus, jaw_positions=jaws, ct_image=1000.0 * x[:, 0, ...], permute_ct=True)
                dose_pred = torch.where(mask_external, dose_pred, torch.zeros_like(dose_pred))

                # Compute loss
                raw_losses = compute_loss(dose_pred, mus, y_mlc, jaws, weights)      # list of tensors
                loss = torch.stack(raw_losses).sum()                   # scalar tensor
                
                # Backprop
                loss.backward()

                # stash anything you want to inspect/plot after step()
                latest["raw_losses"] = [v.detach().item() for v in raw_losses]
                latest["loss_val"]   = loss.detach().item()
                latest["dose_pred"]  = dose_pred.detach()

                return loss

            # --- the actual optimizer step ---
            loss = optimizer.step(closure)   # returns the last loss the closure returned
            # scheduler.step(loss)
            raw_losses = latest["raw_losses"]
            dose_pred = latest["dose_pred"]
            loss_val = latest["loss_val"]
            mae_loss = np.round(torch.mean(torch.abs((y_dose - dose_pred)[masks_torch[-1] > 0])).cpu().detach().numpy(), 4)
            
            patience += 1
            if (mae_loss < current_res[0]):
                patience = 0
                current_res = [mae_loss, weights, y_mlc.cpu().detach().numpy(), mus.cpu().detach().numpy(), jaws.cpu().detach().numpy()]
            else:
                print("Patience count:", patience)
                if ((patience >= patience_thr) | torch.isnan(dose_pred).any()):
                    best_results.append(current_res)
                    print("Best result for this test:", current_res)
                    break
            
            print_results(raw_losses, y_dose, y_mlc, mus, jaws, best_results, dose_pred, masks_torch, mae_loss)
            print(monitor.summary())  # <- per-submodule act + grad stats
            print([result[0] for result in best_results])
    

In [ ]:
np.mean(np.abs(best_results[0][1] - best_results[1][1]))

In [ ]:
best_results[-1][1]

In [ ]:
weights